#### Install Necessary Packages

In [52]:
%pip install pyarrow
%pip install azure-storage-blob


Note: you may need to restart the kernel to use updated packages.
  Using cached azure_storage_blob-12.28.0-py3-none-any.whl.metadata (26 kB)
  Using cached azure_core-1.39.0-py3-none-any.whl.metadata (48 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
Using cached azure_storage_blob-12.28.0-py3-none-any.whl (431 kB)
Using cached azure_core-1.39.0-py3-none-any.whl (218 kB)
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)

   ------------- -------------------------- 1/3 [azure-core]
   ------------- -------------------------- 1/3 [azure-core]
   ------------- -------------------------- 1/3 [azure-core]
   -------------------------- ------------- 2/3 [azure-storage-blob]
   -------------------------- ------------- 2/3 [azure-storage-blob]
   -------------------------- ------------- 2/3 [azure-storage-blob]
   -------------------------- ------------- 2/3 [azure-storage-blob]
   ---------------------------------------- 3/3 [azure-storage-blob]

Note: you may need to r

In [56]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


#### Import Necessary Libraries


In [57]:
import pandas as pd
import os
import io
from azure.storage.blob import BlobServiceClient, BlobClient
from dotenv import load_dotenv

#### Extraction Layer

In [27]:
ziko_df = pd.read_csv('ziko_logistics_data.csv')


In [28]:
ziko_df.head()

,Transaction_ID,Date,Customer_ID,Product_ID,Quantity,Unit_Price,Total_Cost,Discount_Rate,Sales_Channel,Order_Priority,...,Return_Reason,Payment_Type,Taxable,Region,Country,Customer_Name,Customer_Phone,Customer_Email,Customer_Address,Product_List_Title
0,200,2020-01-01 20:32:25.945945945,1086,536,3,120.436821,8265.374549,0.20,Online,High,...,Wrong Item,Wire Transfer,False,West,Canada,Customer 200,+1-652-572-9306,customer.200.78@example.com,"275 Second St, Phoenix, USA",Product 53
1,321,2020-01-02 06:55:08.108108108,1078,523,6,475.724994,4047.850479,NaN,Reseller,Critical,...,Damaged,PayPal,True,South,Mexico,Customer 321,+1-311-186-5760,customer.321.13@sample.com,"478 Third St, New York, USA",Product 33
2,989,2020-01-06 08:12:58.378378378,1077,535,3,146.400556,NaN,0.05,Direct,Critical,...,Damaged,PayPal,True,West,Canada,Customer 989,+1-922-606-9032,customer.989.99@example.com,"843 Second St, Phoenix, USA",Product 6
3,682,2020-01-07 22:03:14.594594594,1027,546,6,19.373194,8194.281993,NaN,Reseller,Medium,...,Wrong Item,Cash,True,South,Canada,Customer 682,+1-237-853-5808,customer.682.66@demo.com,"153 Main St, Phoenix, USA",Product 68
4,484,2020-01-07 07:08:06.486486486,1052,556,8,193.221313,8331.329249,0.20,Direct,Low,...,Late,Cash,False,South,Mexico,Customer 484,+1-986-360-9109,customer.484.3@sample.com,"264 Second St, New York, USA",Product 89


In [29]:
# To get the summary statistics of the DataFrame
ziko_df.describe()

,Transaction_ID,Customer_ID,Product_ID,Quantity,Unit_Price,Total_Cost,Discount_Rate
count,1005.000000,1005.000000,1005.000000,1005.000000,904.000000,905.000000,714.000000
mean,499.057711,1050.042786,529.886567,4.972139,256.467791,5096.553818,0.122479
std,288.867556,29.281420,17.192288,2.555991,142.409717,2688.542137,0.055316
min,1.000000,1001.000000,501.000000,1.000000,10.005701,500.291829,0.050000
25%,248.000000,1025.000000,515.000000,3.000000,134.082539,2727.043200,0.100000
50%,498.000000,1050.000000,530.000000,5.000000,258.425121,5046.019514,0.100000
75%,749.000000,1075.000000,544.000000,7.000000,378.059382,7383.683725,0.150000
max,1000.000000,1100.000000,560.000000,9.000000,499.457380,9966.999934,0.200000


In [30]:
ziko_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Transaction_ID         1005 non-null   int64  
 1   Date                   1005 non-null   object 
 2   Customer_ID            1005 non-null   int64  
 3   Product_ID             1005 non-null   int64  
 4   Quantity               1005 non-null   int64  
 5   Unit_Price             904 non-null    float64
 6   Total_Cost             905 non-null    float64
 7   Discount_Rate          714 non-null    float64
 8   Sales_Channel          1005 non-null   object 
 9   Order_Priority         1005 non-null   object 
 10  Warehouse_Code         1005 non-null   object 
 11  Ship_Mode              1005 non-null   object 
 12  Delivery_Status        1005 non-null   object 
 13  Customer_Satisfaction  1005 non-null   object 
 14  Item_Returned          1005 non-null   bool   
 15  Retu

In [31]:
ziko_df.columns

Index(['Transaction_ID', 'Date', 'Customer_ID', 'Product_ID', 'Quantity',
       'Unit_Price', 'Total_Cost', 'Discount_Rate', 'Sales_Channel',
       'Order_Priority', 'Warehouse_Code', 'Ship_Mode', 'Delivery_Status',
       'Customer_Satisfaction', 'Item_Returned', 'Return_Reason',
       'Payment_Type', 'Taxable', 'Region', 'Country', 'Customer_Name',
       'Customer_Phone', 'Customer_Email', 'Customer_Address',
       'Product_List_Title'],
      dtype='object')

#### Data Cleaning and Transformation

In [32]:
ziko_df.fillna({
    'Unit_Price': ziko_df['Unit_Price'].mean(),
    'Total_Cost': ziko_df['Total_Cost'].mean(),
    'Discount_Rate': 0.0,
    'Return_Reason': 'Unknown'
}, inplace=True)

In [33]:
ziko_df['Date'] = pd.to_datetime(ziko_df['Date'])

In [34]:
ziko_df.info() # Check for missing values after imputation


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Transaction_ID         1005 non-null   int64         
 1   Date                   1005 non-null   datetime64[ns]
 2   Customer_ID            1005 non-null   int64         
 3   Product_ID             1005 non-null   int64         
 4   Quantity               1005 non-null   int64         
 5   Unit_Price             1005 non-null   float64       
 6   Total_Cost             1005 non-null   float64       
 7   Discount_Rate          1005 non-null   float64       
 8   Sales_Channel          1005 non-null   object        
 9   Order_Priority         1005 non-null   object        
 10  Warehouse_Code         1005 non-null   object        
 11  Ship_Mode              1005 non-null   object        
 12  Delivery_Status        1005 non-null   object        
 13  Cus

In [35]:
ziko_df.head()

,Transaction_ID,Date,Customer_ID,Product_ID,Quantity,Unit_Price,Total_Cost,Discount_Rate,Sales_Channel,Order_Priority,...,Return_Reason,Payment_Type,Taxable,Region,Country,Customer_Name,Customer_Phone,Customer_Email,Customer_Address,Product_List_Title
0,200,2020-01-01 20:32:25.945945945,1086,536,3,120.436821,8265.374549,0.20,Online,High,...,Wrong Item,Wire Transfer,False,West,Canada,Customer 200,+1-652-572-9306,customer.200.78@example.com,"275 Second St, Phoenix, USA",Product 53
1,321,2020-01-02 06:55:08.108108108,1078,523,6,475.724994,4047.850479,0.00,Reseller,Critical,...,Damaged,PayPal,True,South,Mexico,Customer 321,+1-311-186-5760,customer.321.13@sample.com,"478 Third St, New York, USA",Product 33
2,989,2020-01-06 08:12:58.378378378,1077,535,3,146.400556,5096.553818,0.05,Direct,Critical,...,Damaged,PayPal,True,West,Canada,Customer 989,+1-922-606-9032,customer.989.99@example.com,"843 Second St, Phoenix, USA",Product 6
3,682,2020-01-07 22:03:14.594594594,1027,546,6,19.373194,8194.281993,0.00,Reseller,Medium,...,Wrong Item,Cash,True,South,Canada,Customer 682,+1-237-853-5808,customer.682.66@demo.com,"153 Main St, Phoenix, USA",Product 68
4,484,2020-01-07 07:08:06.486486486,1052,556,8,193.221313,8331.329249,0.20,Direct,Low,...,Late,Cash,False,South,Mexico,Customer 484,+1-986-360-9109,customer.484.3@sample.com,"264 Second St, New York, USA",Product 89


##### Create the tables
##### - Customer Table

In [36]:
customer = ziko_df[[ 'Customer_ID', 'Customer_Name', 'Customer_Phone', 'Customer_Email', 'Customer_Address']].copy().drop_duplicates().reset_index(drop=True)

customer.head()

,Customer_ID,Customer_Name,Customer_Phone,Customer_Email,Customer_Address
0,1086,Customer 200,+1-652-572-9306,customer.200.78@example.com,"275 Second St, Phoenix, USA"
1,1078,Customer 321,+1-311-186-5760,customer.321.13@sample.com,"478 Third St, New York, USA"
2,1077,Customer 989,+1-922-606-9032,customer.989.99@example.com,"843 Second St, Phoenix, USA"
3,1027,Customer 682,+1-237-853-5808,customer.682.66@demo.com,"153 Main St, Phoenix, USA"
4,1052,Customer 484,+1-986-360-9109,customer.484.3@sample.com,"264 Second St, New York, USA"


In [37]:
customer.shape

(1000, 5)

##### - Product Table

In [42]:
product = ziko_df[['Product_ID', 'Product_List_Title', 'Quantity', 'Unit_Price', 'Discount_Rate']].copy().drop_duplicates().reset_index(drop=True)

product.head()

,Product_ID,Product_List_Title,Quantity,Unit_Price,Discount_Rate
0,536,Product 53,3,120.436821,0.20
1,523,Product 33,6,475.724994,0.00
2,535,Product 6,3,146.400556,0.05
3,546,Product 68,6,19.373194,0.00
4,556,Product 89,8,193.221313,0.20


In [43]:
product.shape

(1000, 5)

In [44]:
ziko_df.columns

Index(['Transaction_ID', 'Date', 'Customer_ID', 'Product_ID', 'Quantity',
       'Unit_Price', 'Total_Cost', 'Discount_Rate', 'Sales_Channel',
       'Order_Priority', 'Warehouse_Code', 'Ship_Mode', 'Delivery_Status',
       'Customer_Satisfaction', 'Item_Returned', 'Return_Reason',
       'Payment_Type', 'Taxable', 'Region', 'Country', 'Customer_Name',
       'Customer_Phone', 'Customer_Email', 'Customer_Address',
       'Product_List_Title'],
      dtype='object')

##### - Transaction_Fact_Table

In [46]:
transaction_fact = ziko_df.merge(customer, on=['Customer_ID', 'Customer_Name', 'Customer_Phone', 'Customer_Email', 'Customer_Address'], how='left') \
                          .merge(product, on=['Product_ID', 'Product_List_Title', 'Quantity', 'Unit_Price', 'Discount_Rate'], how='left') \
                          [['Transaction_ID', 'Date', 'Customer_ID', 'Product_ID', 'Total_Cost', 'Sales_Channel','Order_Priority',\
                            'Warehouse_Code', 'Ship_Mode', 'Delivery_Status', 'Customer_Satisfaction', 'Item_Returned', 'Return_Reason',\
                            'Payment_Type', 'Taxable', 'Region', 'Country']]

In [47]:
transaction_fact.head()

,Transaction_ID,Date,Customer_ID,Product_ID,Total_Cost,Sales_Channel,Order_Priority,Warehouse_Code,Ship_Mode,Delivery_Status,Customer_Satisfaction,Item_Returned,Return_Reason,Payment_Type,Taxable,Region,Country
0,200,2020-01-01 20:32:25.945945945,1086,536,8265.374549,Online,High,WH-3,2-Day,Cancelled,Neutral,False,Wrong Item,Wire Transfer,False,West,Canada
1,321,2020-01-02 06:55:08.108108108,1078,523,4047.850479,Reseller,Critical,WH-1,Overnight,Backorder,Satisfied,True,Damaged,PayPal,True,South,Mexico
2,989,2020-01-06 08:12:58.378378378,1077,535,5096.553818,Direct,Critical,WH-1,Overnight,Pending,Unsatisfied,True,Damaged,PayPal,True,West,Canada
3,682,2020-01-07 22:03:14.594594594,1027,546,8194.281993,Reseller,Medium,WH-1,Express,Pending,Unsatisfied,False,Wrong Item,Cash,True,South,Canada
4,484,2020-01-07 07:08:06.486486486,1052,556,8331.329249,Direct,Low,WH-2,2-Day,Delivered,Satisfied,True,Late,Cash,False,South,Mexico


#### Temporary Loading to CSV (In practice, this would be loaded to a database or data warehouse)

In [48]:
customer.to_csv(r'dataset/customer.csv', index=False)
product.to_csv(r'dataset/product.csv', index=False)
transaction_fact.to_csv(r'dataset/transaction_fact.csv', index=False)

#### Data Loading

In [ ]:
#Azure Blob Connection
# Define your Azure Blob Storage connection string and container name

connection_str = "xxxxxxxx"
blob_service_client = BlobServiceClient.from_connection_string(connection_str)


container_name = "xxxxxxxxxx"
container_client = blob_service_client.get_container_client(container_name)

In [58]:
# Azure Blob Connection
# Define your Azure Blob Storage connection string and container name

load_dotenv()  # Load environment variables from .env file

connection_str = os.getenv("CONNECTION_STR")
blob_service_client = BlobServiceClient.from_connection_string(connection_str)


container_name = os.getenv("CONTAINER_NAME")
container_client = blob_service_client.get_container_client(container_name)

In [60]:
# Create a function to load the DataFrame to Azure Blob Storage as a parquet file
def upload_df_to_bob_as_parquet(df, container_client, blob_name):

    # Convert DataFrame to parquet format in memory
    parquet_buffer = io.BytesIO()
    df.to_parquet(parquet_buffer, index=False)
    parquet_buffer.seek(0)  # Move the buffer position to the beginning
    
    # Create a BlobClient and upload the parquet file
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(parquet_buffer, blob_type="BlockBlob", overwrite=True)
    print(f"DataFrame successfully uploaded to Azure Blob Storage as {blob_name}")

In [61]:
upload_df_to_bob_as_parquet(customer, container_client, 'rawdata/customer.parquet')
upload_df_to_bob_as_parquet(product, container_client, 'rawdata/product.parquet')
upload_df_to_bob_as_parquet(transaction_fact, container_client, 'rawdata/transaction_fact.parquet')

DataFrame successfully uploaded to Azure Blob Storage as rawdata/customer.parquet
DataFrame successfully uploaded to Azure Blob Storage as rawdata/product.parquet
DataFrame successfully uploaded to Azure Blob Storage as rawdata/transaction_fact.parquet



#### 